In [52]:
from google.cloud import bigquery

client = bigquery.Client(project="prod-organize-arizon-4e1c0a83")

query = """
SELECT
*
FROM `prod-organize-arizon-4e1c0a83.viewers_dataset.az_censustract_voters_2024`
"""


df = client.query(query).to_dataframe()

df.head()

,mailaddrcensustract20,geoid,population_count,dem_votes,rep_votes,dem_margin,third_votes
0,812200,040138122004015,8344,1089,1669,0.652487,1386
1,092724,040130927241001,4930,610,639,0.954617,672
2,030507,120690305074052,872,108,218,0.495413,210
3,391000,060133910002009,134,44,32,1.375000,21
4,082213,040130822131017,5368,784,332,2.361446,529


In [53]:
import geopandas as gpd

# so didn't know you can do this with raw url until today >:)
url = "https://raw.githubusercontent.com/cmarikos/az-landscape-2024/christina-dev/Analysis%20Files/az_census_tract.geojson"
gdf = gpd.read_file(url)


gdf.crs

<Geographic 2D CRS: EPSG:4269>
Name: NAD83
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: North America - onshore and offshore: Canada - Alberta; British Columbia; Manitoba; New Brunswick; Newfoundland and Labrador; Northwest Territories; Nova Scotia; Nunavut; Ontario; Prince Edward Island; Quebec; Saskatchewan; Yukon. Puerto Rico. United States (USA) - Alabama; Alaska; Arizona; Arkansas; California; Colorado; Connecticut; Delaware; Florida; Georgia; Hawaii; Idaho; Illinois; Indiana; Iowa; Kansas; Kentucky; Louisiana; Maine; Maryland; Massachusetts; Michigan; Minnesota; Mississippi; Missouri; Montana; Nebraska; Nevada; New Hampshire; New Jersey; New Mexico; New York; North Carolina; North Dakota; Ohio; Oklahoma; Oregon; Pennsylvania; Rhode Island; South Carolina; South Dakota; Tennessee; Texas; Utah; Vermont; Virginia; Washington; West Virginia; Wisconsin; Wyoming. US Virgin Islands. British Virgin Islands

In [54]:
# above gdf.crs told me that we're in EPSG:4269
# folium uses EPSG:4326 so we gotta convert

gdf = gdf.to_crs(4326)

In [55]:
len(df["geoid"])

45648

In [56]:
df.head(10)

,mailaddrcensustract20,geoid,population_count,dem_votes,rep_votes,dem_margin,third_votes
0,812200,040138122004015,8344,1089,1669,0.652487,1386
1,092724,040130927241001,4930,610,639,0.954617,672
2,030507,120690305074052,872,108,218,0.495413,210
3,391000,060133910002009,134,44,32,1.375000,21
4,082213,040130822131017,5368,784,332,2.361446,529
5,004201,530330042011000,1369,350,218,1.605505,307
6,031203,120190312031009,1072,146,225,0.648889,219
7,010307,191030103071022,1565,206,408,0.504902,371
8,020811,450150208113000,167,40,46,0.869565,27
9,204500,390852045002007,74,37,5,7.400000,20


In [57]:
import pandas as pd
import numpy as np

twoparty = df[["dem_votes", "rep_votes"]].sum(axis=1, skipna=True)
with np.errstate(divide='ignore', invalid='ignore'):
    df["dem_margin_2p"] = np.where(twoparty > 0,
                                    ( df["rep_votes"] - df["dem_votes"]) / twoparty,
                                    np.nan)

# this made my geoids not unique anymore
# no idea what the last four digits are here
# if the census bureau doesn't use them then eh
chop_four = df["tract_geoid11"] = (
    df["geoid"].astype("string")
      .str.replace(r"\D", "", regex=True)
      .str.zfill(15)
      .str[:11]
)

df.head(5)

,mailaddrcensustract20,geoid,population_count,dem_votes,rep_votes,dem_margin,third_votes,dem_margin_2p,tract_geoid11
0,812200,040138122004015,8344,1089,1669,0.652487,1386,0.210297,04013812200
1,092724,040130927241001,4930,610,639,0.954617,672,0.023219,04013092724
2,030507,120690305074052,872,108,218,0.495413,210,0.337423,12069030507
3,391000,060133910002009,134,44,32,1.375000,21,-0.157895,06013391000
4,082213,040130822131017,5368,784,332,2.361446,529,-0.405018,04013082213


In [58]:
gdf.head(10)

,STATEFP,COUNTYFP,TRACTCE,GEOID,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
0,04,015,950500,04015950500,9505,Census Tract 9505,G5020,S,1255016476,39469850,+35.3440369,-114.3629894,"POLYGON ((-114.67888 35.50137, -114.67883 35.5..."
1,04,015,954900,04015954900,9549,Census Tract 9549,G5020,S,22392503,0,+35.2440284,-114.0595108,"POLYGON ((-114.10394 35.26982, -114.10256 35.2..."
2,04,015,951800,04015951800,9518,Census Tract 9518,G5020,S,3648437,120729,+35.0963207,-114.6188545,"POLYGON ((-114.64029 35.09852, -114.64012 35.0..."
3,04,015,952400,04015952400,9524,Census Tract 9524,G5020,S,295669288,7885448,+34.6247879,-114.3458669,"POLYGON ((-114.48778 34.71722, -114.48622 34.7..."
4,04,027,012100,04027012100,121,Census Tract 121,G5020,S,6508727795,60851,+33.1077362,-113.8376677,"POLYGON ((-114.47325 33.02788, -114.45989 33.0..."
5,04,027,000200,04027000200,2,Census Tract 2,G5020,S,1604889,15347,+32.7188025,-114.6288918,"POLYGON ((-114.63352 32.72563, -114.63337 32.7..."
6,04,027,000301,04027000301,3.01,Census Tract 3.01,G5020,S,3313611,141941,+32.7270014,-114.6493156,"POLYGON ((-114.66752 32.7243, -114.66751 32.72..."
7,04,027,000908,04027000908,9.08,Census Tract 9.08,G5020,S,23008504,4187,+32.6474716,-114.6627542,"POLYGON ((-114.68464 32.66928, -114.68461 32.6..."
8,04,027,011000,04027011000,110,Census Tract 110,G5020,S,68606226,875964,+32.6737450,-114.7159650,"POLYGON ((-114.77 32.63181, -114.76998 32.6318..."
9,04,027,980003,04027980003,9800.03,Census Tract 9800.03,G5020,S,1843805607,139619,+32.2428436,-113.5327647,"POLYGON ((-113.96145 32.24504, -113.96131 32.2..."


In [59]:
import numpy as np
# Group by 'tract_geoid11' and sum the relevant columns
metrics = df.groupby("tract_geoid11").agg(
    population_count=("population_count", "sum"),
    dem_votes=("dem_votes", "sum"),
    rep_votes=("rep_votes", "sum"),
    third_votes=("third_votes", "sum")
).reset_index()

# Recalculate dem_margin_2p after aggregation
twoparty_grouped = metrics["dem_votes"] + metrics["rep_votes"]
with np.errstate(divide='ignore', invalid='ignore'):
    metrics["dem_margin_2p"] = np.where(twoparty_grouped > 0,
                                    (metrics["dem_votes"] - metrics["rep_votes"]) / twoparty_grouped,
                                    np.nan)

gdf = gdf.merge(metrics, left_on="GEOID", right_on="tract_geoid11", how="left")


In [60]:
gdf.head(5)

,STATEFP,COUNTYFP,TRACTCE,GEOID,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry,tract_geoid11,population_count,dem_votes,rep_votes,third_votes,dem_margin_2p
0,04,015,950500,04015950500,9505,Census Tract 9505,G5020,S,1255016476,39469850,+35.3440369,-114.3629894,"POLYGON ((-114.67888 35.50137, -114.67883 35.5...",04015950500,30496,1498,6873,2684,-0.642098
1,04,015,954900,04015954900,9549,Census Tract 9549,G5020,S,22392503,0,+35.2440284,-114.0595108,"POLYGON ((-114.10394 35.26982, -114.10256 35.2...",04015954900,27169,1598,5371,2670,-0.541398
2,04,015,951800,04015951800,9518,Census Tract 9518,G5020,S,3648437,120729,+35.0963207,-114.6188545,"POLYGON ((-114.64029 35.09852, -114.64012 35.0...",04015951800,25541,1576,4695,2222,-0.497369
3,04,015,952400,04015952400,9524,Census Tract 9524,G5020,S,295669288,7885448,+34.6247879,-114.3458669,"POLYGON ((-114.48778 34.71722, -114.48622 34.7...",04015952400,25358,1133,6318,2023,-0.695880
4,04,027,012100,04027012100,121,Census Tract 121,G5020,S,6508727795,60851,+33.1077362,-113.8376677,"POLYGON ((-114.47325 33.02788, -114.45989 33.0...",04027012100,8827,914,2281,1386,-0.427856


In [ ]:
import folium
import numpy as np
import pandas as pd
from branca.colormap import LinearColormap
from folium.features import GeoJson, GeoJsonTooltip


# ============================================
KEY_COL  = "GEOID"
POP_COL  = "population_count"
DEM_COL  = "dem_votes"
REP_COL  = "rep_votes"
OTH_COL  = "third_votes"
MARGIN_COL = "dem_margin_2p" # Changed from "dem_margin_2p_y"
# ============================================

# Filter gdf to include only Arizona (STATEFP '04')
gdf = gdf[gdf['STATEFP'] == '04']

minx, miny, maxx, maxy = gdf.total_bounds
m = folium.Map(location=[(miny+maxy)/2, (minx+maxx)/2], zoom_start=6, tiles="CartoDB Positron")

abs_margin = gdf[MARGIN_COL].abs().dropna() # Use MARGIN_COL
vmax = float(np.quantile(abs_margin, 0.98)) if len(abs_margin) else 1.0
cmap = LinearColormap(colors=["#2166ac", "#f7f7f7", "#b2182b"], vmin=-vmax, vmax=vmax)
cmap.caption = "Dem margin (two-party, Dem − Rep)"
cmap.add_to(m)


def style_fn(feat):
    val = feat["properties"].get(MARGIN_COL) # Use MARGIN_COL
    if pd.isna(val):
        return {"fillColor": "#cccccc", "fillOpacity": 0.25, "weight": 0.4, "color": "#666"}
    return {"fillColor": cmap(val), "fillOpacity": 0.8, "weight": 0.4, "color": "#666"}

tooltip_fields = [c for c in [KEY_COL, MARGIN_COL, POP_COL] if c in gdf.columns] # Use MARGIN_COL
tooltip_aliases = ["Pct:", "Dem margin (2-party):", "Population:"]

poly_layer = GeoJson(
    data=gdf.to_json(),
    name="Census tracts 2020 — Dem margin",
    style_function=style_fn,
    highlight_function=lambda f: {"weight": 2, "color": "#000"},
    tooltip=GeoJsonTooltip(fields=tooltip_fields, aliases=tooltip_aliases, localize=True, sticky=False),
)
poly_layer.add_to(m)

# Population “spikes” as scaled circle markers at representative points
# Population “spikes” as scaled circle markers at representative points
pop_layer = folium.FeatureGroup(name="Population spikes", show=False)

max_pop = gdf[POP_COL].max() if POP_COL in gdf.columns else None

def pop_radius(pop, min_r=3, max_r=18):
    if (pop is None) or pd.isna(pop) or (max_pop is None) or (max_pop <= 0):
        return 0
    return float(min_r + (max_r - min_r) * np.sqrt(pop / max_pop))

# --- SAFER: filter to valid geometries, compute points, drop null points & pops
gdf_pts = gdf[gdf.geometry.notna() & ~gdf.geometry.apply(lambda g: getattr(g, "is_empty", True))].copy()
gdf_pts["__pt"] = gdf_pts.geometry.apply(lambda g: g.representative_point() if (g is not None and not g.is_empty) else None)
gdf_pts = gdf_pts[gdf_pts["__pt"].notna() & gdf_pts[POP_COL].notna()].copy()

for _, row in gdf_pts.iterrows():
    r = pop_radius(row[POP_COL])
    if r <= 0:
        continue
    lat, lon = float(row["__pt"].y), float(row["__pt"].x)
    folium.CircleMarker(
        location=[lat, lon],
        radius=r,
        weight=0.8,
        color="#222",
        fill=True,
        fill_opacity=0.45,
        fill_color="#ffffff",
        popup=folium.Popup(
            f"Pct: {row.get(KEY_COL,'')}"
            + (f"<br>Population: {int(row[POP_COL]):,}" if pd.notna(row[POP_COL]) else "")
            + (f"<br>Dem margin (2p): {row[MARGIN_COL]:.3f}" if pd.notna(row.get(MARGIN_COL)) else ""), # Use MARGIN_COL
            max_width=320
        ),
    ).add_to(pop_layer)

pop_layer.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m.fit_bounds([[miny, minx], [maxy, maxx]])
m.save("az_precincts_demmargin_popspikes.html")

In [ ]:
from IPython.display import IFrame
IFrame("map.html", width="100%", height=600)

m